# Chapter 9: Data Quality with SHACL

**Level 2: Modeling & Querying** · ⏱ about 2.5 hours · Dataset: *Recipes & Nutrition*

### What you will learn

1. Why a knowledge graph needs **closed-world validation** next to open-world reasoning
2. The anatomy of **SHACL shapes**: targets, property paths, constraints, severities, messages
3. Running **validation reports** from Python and turning them into tables
4. **SPARQL-based constraints** for rules no single property can express
5. **Hands-on:** a **validation gate** that rejects a bad batch before it reaches the graph
6. **Mini-project 2:** an **allergen-safe menu service**: validated data + reasoning answering "what can this guest eat?"

In [1]:
import sys
from pathlib import Path

SRC = str(Path('..', 'src').resolve())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pandas as pd
import stardog
from rdflib import RDF

from kg import client, models, reasoning, recipes, sparql, validation
from kg.recipes import ALLERGEN, REC, RECIPE

pd.set_option('display.max_colwidth', 75)
pd.set_option('display.width', 200)
conn = client.connect()
PREFIX = 'urn:tutorial:ch09'
DATA, MODEL, SHAPES = f'{PREFIX}:data', f'{PREFIX}:model', f'{PREFIX}:shapes'
MODEL_NAME = 'RecipeKG'
ROOT = Path('..')

def reload_data():
    with client.transaction(conn):
        conn.clear(graph_uri=DATA)
        conn.add(stardog.content.File(str(ROOT / 'data' / 'recipes' / 'recipes.ttl')), graph_uri=DATA)

with client.transaction(conn) as stats:
    for g in (MODEL, SHAPES):
        conn.clear(graph_uri=g)
    conn.add(stardog.content.File(str(ROOT / 'ontology' / 'recipes.ttl')), graph_uri=MODEL)
    conn.add(stardog.content.File(str(ROOT / 'ontology' / 'recipes_rules.ttl')), graph_uri=MODEL)
    conn.add(stardog.content.File(str(ROOT / 'ontology' / 'recipes_shapes.ttl')), graph_uri=SHAPES)
reload_data()
models.register_model(MODEL_NAME, MODEL)
print('Loaded model, rules, shapes and data')

PREFIXES = '''PREFIX rec: <http://example.org/recipes#>
PREFIX recipe: <http://example.org/recipes/recipe/>
PREFIX ingredient: <http://example.org/recipes/ingredient/>
PREFIX category: <http://example.org/recipes/category/>
'''
def short(iri):
    return str(iri).rsplit('/', 1)[-1]

Loaded model, rules, shapes and data


Three graphs this time: **data**, **model** (ontology + rules), and **shapes**, the SHACL rules of this chapter, kept in their own graph like the model.

## 9.1 Why validation?

Chapter 8's reasoner works under the **open world** assumption. A recipe without a name is fine ("the name just isn't recorded"), and an unknown ingredient simply becomes a new, untyped thing. That's right for **inference**, but a menu app needs guarantees: every recipe **has** a name, **exactly one** prep time, only **known** ingredients.

**SHACL** (Shapes Constraint Language, a W3C standard) checks data under the **closed world** assumption: what isn't there is missing.

| | OWL / rules (Ch 7–8) | SHACL (this chapter) |
|---|---|---|
| Question | What **follows** from the data? | Does the data **meet the requirements**? |
| World | Open: missing = unknown | Closed: missing = missing |
| `minCount 1` / "has a name" | Can't be checked | ✅ |
| "Exactly one value" | Merges or contradicts | ✅ reported with a message |
| Output | New triples | A **validation report** |

They complement each other: **validate** what comes in, then **reason** over what's valid.

> 🔁 **Neo4j equivalent:** Neo4j constraints cover uniqueness, existence, and (in newer versions) property types. SHACL goes much further: value ranges, patterns, allowed lists, cardinalities, links to typed nodes, and arbitrary SPARQL checks. The neosemantics plugin can run SHACL over Neo4j.

## 9.2 Anatomy of a shape

```turtle
rec:RecipeShape a sh:NodeShape ;                 # a shape …
    sh:targetClass rec:Recipe ;                  # … applies to every rec:Recipe (its "focus nodes")
    sh:property [                                # a property shape
        sh:path rec:prepMinutes ;                #   what to check: the values of this path
        sh:minCount 1 ; sh:maxCount 1 ;          #   constraints
        sh:datatype xsd:integer ;
        sh:minInclusive 1 ; sh:maxInclusive 600 ;
        sh:severity sh:Violation ;               #   how bad (default: Violation)
        sh:message "prepMinutes: exactly one whole number, 1-600" ] .
```

**Targets**: which nodes to check. `sh:targetClass` (all instances), `sh:targetNode` (specific ones), `sh:targetSubjectsOf` / `sh:targetObjectsOf` (everything using a property).

**Constraints** used in this chapter:

| Constraint | Checks |
|---|---|
| `sh:minCount`, `sh:maxCount` | How many values: required, at most one, … |
| `sh:datatype` | Literal datatype, e.g. `xsd:integer` |
| `sh:class` | Values are instances of a class, e.g. a known `rec:Ingredient` |
| `sh:minInclusive`, `sh:maxInclusive`, `sh:minExclusive` | Numeric ranges |
| `sh:pattern` | A regular expression for text |
| `sh:in` | One of a fixed list |
| `sh:uniqueLang`, `sh:languageIn` | At most one value per language, allowed languages |
| `sh:qualifiedValueShape` + `sh:qualifiedMinCount` | "At least N values that look like …" (e.g. an English name) |
| `sh:node` | Each value must itself conform to another shape |
| `sh:sparql` | Any condition written as a SPARQL query (9.6) |

**Severities**: `sh:Violation` (default), `sh:Warning`, `sh:Info`. They're labels; *you* decide what each one means for your pipeline.

### See every constraint fire

A tiny demo: one shape, and three recipes that break it in different ways:

In [2]:
DEMO_SHAPES, DEMO_DATA = f'{PREFIX}:demo:shapes', f'{PREFIX}:demo:data'
with client.transaction(conn):
    for g in (DEMO_SHAPES, DEMO_DATA):
        conn.clear(graph_uri=g)
    conn.add(stardog.content.Raw(b'''
@prefix sh: <http://www.w3.org/ns/shacl#> . @prefix rec: <http://example.org/recipes#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
rec:DemoShape a sh:NodeShape ; sh:targetClass rec:Recipe ;
  sh:property [ sh:path rec:name ; sh:minCount 1 ; sh:message "needs a name" ] ;
  sh:property [ sh:path rec:prepMinutes ; sh:maxCount 1 ; sh:datatype xsd:integer ; sh:minInclusive 1 ;
                sh:message "prepMinutes: one positive integer" ] ;
  sh:property [ sh:path rec:recipeId ; sh:pattern "^R[0-9]{2,}$" ; sh:message "recipeId like R15" ] ;
  sh:property [ sh:path rec:cuisine ; sh:in ("Italian" "Thai") ; sh:severity sh:Warning ;
                sh:message "unknown cuisine" ] .
''', 'text/turtle'), graph_uri=DEMO_SHAPES)
    conn.add(stardog.content.Raw(b'''
@prefix rec: <http://example.org/recipes#> . @prefix recipe: <http://example.org/recipes/recipe/> .
recipe:ok      a rec:Recipe ; rec:name "Fine" ; rec:prepMinutes 10 ; rec:recipeId "R01" ; rec:cuisine "Thai" .
recipe:no-name a rec:Recipe ; rec:prepMinutes 10 , 12 ; rec:recipeId "R02" ; rec:cuisine "Italian" .
recipe:messy   a rec:Recipe ; rec:name "Messy" ; rec:prepMinutes "ten" ; rec:recipeId "recipe-3" ; rec:cuisine "Thia" .
''', 'text/turtle'), graph_uri=DEMO_DATA)

demo = validation.validate(conn, DEMO_SHAPES, [DEMO_DATA])
print('conforms:', demo.conforms, '|', demo.summary())
demo.results.assign(focus=demo.results.focus.map(short))

conforms: False | 5 violation(s), 1 warning(s), 0 info(s)


,severity,focus,path,value,message,constraint
0,Violation,messy,prepMinutes,ten,prepMinutes: one positive integer,Datatype
1,Violation,messy,prepMinutes,ten,prepMinutes: one positive integer,MinInclusive
2,Violation,messy,recipeId,recipe-3,recipeId like R15,Pattern
3,Violation,no-name,name,,needs a name,MinCount
4,Violation,no-name,prepMinutes,,prepMinutes: one positive integer,MaxCount
5,Warning,messy,cuisine,Thia,unknown cuisine,In


Every finding names the **focus node** (which resource), the **path** (which property), the offending **value**, the **message**, and the **constraint** that failed. `recipe:ok` has no findings.

Note `"Thia"`: a typo caught by `sh:in`, reported as a **Warning** because we marked it so.

## 9.3 Validation from Python

pystardog offers `conn.icv().report(...)`, which returns the standard **SHACL validation report** as Turtle. `kg.validation.validate()` calls the same endpoint, but lets you pass **several** data graphs (needed in 9.7), and parses the report into the table above:

In [3]:
body = '\n'.join(l for l in demo.turtle.splitlines() if l.strip() and not l.startswith('@prefix'))
print(body[:1100], '\n…')

_:bnode_1fa8db8c_e827_4dee_8261_1a88cd4948f4_3421 a sh:ValidationReport ;
   sh:conforms false .
_:bnode_1fa8db8c_e827_4dee_8261_1a88cd4948f4_3422 a sh:ValidationResult ;
   sh:resultSeverity sh:Violation ;
   sh:sourceShape _:anon-genid-02606754e6aa44cd93c24bab240016c5-10 ;
   sh:sourceConstraintComponent sh:MinCountConstraintComponent ;
   sh:focusNode <http://example.org/recipes/recipe/no-name> ;
   sh:resultPath <http://example.org/recipes#name> ;
   sh:resultMessage "needs a name" .
_:bnode_1fa8db8c_e827_4dee_8261_1a88cd4948f4_3421 sh:result _:bnode_1fa8db8c_e827_4dee_8261_1a88cd4948f4_3422 .
_:bnode_1fa8db8c_e827_4dee_8261_1a88cd4948f4_3423 a sh:ValidationResult ;
   sh:resultSeverity sh:Violation ;
   sh:sourceShape _:anon-genid-02606754e6aa44cd93c24bab240016c5-11 ;
   sh:sourceConstraintComponent sh:MaxCountConstraintComponent ;
   sh:focusNode <http://example.org/recipes/recipe/no-name> ;
   sh:resultPath <http://example.org/recipes#prepMinutes> ;
   sh:resultMessage "prepMinu

In [4]:
print(Path(validation.__file__).read_text()[:2400], '\n…')

"""SHACL validation with Stardog, and a validation gate for loading data.

    report = validation.validate(conn, SHAPES_GRAPH, [DATA_GRAPH])
    report.ok, report.results            # no violations?  DataFrame of findings

    report = validation.load_validated(conn, stardog.content.File('batch.ttl'),
                                       target=DATA_GRAPH, shapes=SHAPES_GRAPH)
"""
from dataclasses import dataclass

import pandas as pd
import stardog
from rdflib import BNode, Graph, Namespace

from kg import client

SH = Namespace('http://www.w3.org/ns/shacl#')
COLUMNS = ['severity', 'focus', 'path', 'value', 'message', 'constraint']


@dataclass
class Report:
  conforms: bool              # SHACL's verdict: False if there is ANY result, even a warning
  results: pd.DataFrame       # one row per finding
  turtle: str = ''            # the raw SHACL validation report

  def count(self, severity: str) -> int:
    return int((self.results.severity == severity).sum())

  @property
  def 

### `conforms` vs `ok`

SHACL's `sh:conforms` is `false` as soon as there is **any** result, even a single Info. That's too strict for a pipeline, so `Report.ok` means **"no Violations"**: warnings and infos are allowed through, but reported.

## 9.4 The Recipes shapes

The real shapes live in `ontology/recipes_shapes.ttl`: recipes, ingredient lines, ingredients and ratings.

In [5]:
print((ROOT / 'ontology' / 'recipes_shapes.ttl').read_text(encoding='utf-8')[:4200], '\n…')

# SHACL shapes for the Recipes dataset (Chapter 9).
# Closed-world data quality rules: what valid data must look like.
#
# Severities: sh:Violation (default) = reject, sh:Warning = accept but flag, sh:Info = nice to know.

@prefix sh:   <http://www.w3.org/ns/shacl#> .
@prefix rec:  <http://example.org/recipes#> .
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

#################################################################
# Recipes
#################################################################

rec:RecipeShape a sh:NodeShape ;
    rdfs:label "Recipe" ;
    sh:targetClass rec:Recipe ;

    sh:property [
        sh:path rec:recipeId ;
        sh:minCount 1 ; sh:maxCount 1 ;
        sh:pattern "^R[0-9]{2,}$" ;
        sh:message "recipeId: exactly one, like R15" ] ;

    sh:property [
        sh:path rec:name ;
        sh:minCount 1 ;
        sh:uniqueLang true 

Design choices worth copying:

- **Every constraint has a `sh:message`** written for the person who has to fix the data.
- **Several property shapes on one path** (`rec:name`, `rec:cuisine`, `rec:created`) separate a hard rule (Violation) from a soft one (Warning or Info).
- **Ingredient lines are checked through the recipe** (`sh:node rec:IngredientLineShape`), so every problem is reported against a recipe IRI. That's easier to act on than a blank node. The gate in 9.7 relies on it.

### Validate the dataset

In [6]:
report = validation.validate(conn, SHAPES, [DATA])
print('conforms:', report.conforms, '|', report.summary())

conforms: True | 0 violation(s), 0 warning(s), 0 info(s)


The data from Chapter 3 is clean. Now damage a few things, the way real data goes wrong:

In [7]:
conn.update(PREFIXES + f'''
WITH <{DATA}>
DELETE {{ recipe:pad-thai rec:name ?en }}                       # lose the English name
INSERT {{ recipe:pesto-pasta rec:prepMinutes 45 .                # a second prep time
         ingredient:tofu rec:proteinPer100g -8.0 .              # a negative value (and a second one)
         recipe:pancakes rec:cuisine "Amercian" .               # a typo, and a second cuisine
         recipe:pancakes rec:hasIngredient ingredient:dragon-fruit . }}  # an unknown ingredient
WHERE  {{ recipe:pad-thai rec:name ?en  FILTER(lang(?en) = "en") }}''')

report = validation.validate(conn, SHAPES, [DATA])
print(report.summary())
report.results.assign(focus=report.results.focus.map(short))

6 violation(s), 1 warning(s), 0 info(s)


,severity,focus,path,value,message,constraint
0,Violation,tofu,proteinPer100g,-8.0,"proteinPer100g: exactly one decimal, 0-100",MinInclusive
1,Violation,tofu,proteinPer100g,,"proteinPer100g: exactly one decimal, 0-100",MaxCount
2,Violation,pad-thai,name,,name: an English name is required,QualifiedMinCount
3,Violation,pancakes,cuisine,,"cuisine: exactly one, as plain text",MaxCount
4,Violation,pancakes,hasIngredient,http://example.org/recipes/ingredient/dragon-fruit,"hasIngredient: at least one, and each must be a known rec:Ingredient",Class
5,Violation,pesto-pasta,prepMinutes,,"prepMinutes: exactly one whole number of minutes, 1-600",MaxCount
6,Warning,pancakes,cuisine,Amercian,"cuisine: not in the known list (a typo, or a new cuisine to add?)",In


Every problem is found, with a clear message. One defect can trigger **more than one** finding: Tofu's extra protein value breaks both "exactly one value" and "0–100". The made-up `dragon-fruit` is caught by `sh:class`: it isn't a known `rec:Ingredient`.

Put the data back:

In [8]:
reload_data()
print(validation.validate(conn, SHAPES, [DATA]).summary())

0 violation(s), 0 warning(s), 0 info(s)


## 9.5 Severity in practice

Severities are just labels. A typical policy:

| Severity | Meaning | Pipeline action |
|---|---|---|
| **Violation** | The data is wrong or unusable | **Reject** the batch |
| **Warning** | Suspicious, probably needs a person | Accept, notify the data owner |
| **Info** | Nice to have | Accept, include in a quality dashboard |

Focus on the questions the report answers for a data steward: *how many problems, of which kind, where?*

In [9]:
conn.update(PREFIXES + f'''WITH <{DATA}> DELETE {{ ?r rec:description ?d ; rec:created ?c }}
WHERE {{ ?r rec:cuisine "Thai" ; rec:description ?d ; rec:created ?c }}''')     # drop some optional fields

report = validation.validate(conn, SHAPES, [DATA])
print('ok for loading?', report.ok, '|', report.summary())
display(report.results.groupby(['severity', 'path', 'message']).size().rename('count').reset_index())
reload_data()

ok for loading? True | 0 violation(s), 2 warning(s), 2 info(s)


,severity,path,message,count
0,Info,created,created: no creation date,2
1,Warning,description,description: missing,2


Missing descriptions (Warning) and dates (Info) don't block anything: `ok` is `True`, even though `conforms` is `False`.

## 9.6 SPARQL-based constraints

Some rules involve **several properties at once**. Our data lists ingredients twice, in `rec:hasIngredient` and in the ingredient lines (Chapter 3), and the two must agree. No single-property constraint can say that. A `sh:sparql` constraint can: its query returns the **violations**, with `$this` bound to each focus node:

```turtle
sh:sparql [
    a sh:SPARQLConstraint ;
    sh:message "an ingredient line uses an ingredient that is missing from hasIngredient" ;
    sh:select '''
        SELECT $this ?value WHERE {
          $this rec:hasIngredientLine/rec:ingredient ?value .
          FILTER NOT EXISTS { $this rec:hasIngredient ?value }
        }''' ] .
```

In [10]:
conn.update(PREFIXES + f'''WITH <{DATA}>
DELETE {{ recipe:pad-thai rec:hasIngredient ingredient:lime }} WHERE {{ recipe:pad-thai rec:hasIngredient ingredient:lime }}''')
report = validation.validate(conn, SHAPES, [DATA])
display(report.results.assign(focus=report.results.focus.map(short)))
reload_data()

,severity,focus,path,value,message,constraint
0,Violation,pad-thai,,http://example.org/recipes/ingredient/lime,an ingredient line uses an ingredient that is missing from hasIngredient,SPARQL


Anything you can write as a query, you can turn into a constraint: cross-field checks, totals, dates in the right order, …

> **SHACL and reasoning.** `sh:targetClass` also covers **sub-classes** declared in the shapes graph (`rdfs:subClassOf`). Validation can run **with reasoning** too (the report endpoint takes `reasoning=true`), so inferred types and links count. Keep it off for ingest checks: you usually want to validate what was actually **sent**.

## 9.7 Hands-on: a validation gate

New recipes arrive from a CMS as a **batch**. Rule: **a batch with any Violation is rejected as a whole**; nothing of it reaches the graph. Warnings and infos pass but are reported.

`validation.load_validated()` implements the gate:

1. load the batch into a **staging graph**, never into the real one;
2. validate staging **together with** the target graph, because new recipes refer to ingredients that already exist, and `sh:class rec:Ingredient` needs to see them;
3. keep only findings about resources **in the batch**;
4. no Violations → `ADD <staging> TO <target>` in one transaction; otherwise leave the target untouched;
5. always drop the staging graph.

In [11]:
def batch(*recipe_dicts):
    '''Several recipes (kg.recipes dict format) as one Turtle upload.'''
    g = recipes.new_graph()
    for r in recipe_dicts:
        g += recipes.recipe_to_graph(r)
    return stardog.content.Raw(g.serialize(format='turtle').encode(), 'text/turtle')

def recipe_count():
    return int(sparql.run_query(conn, 'PREFIX rec: <http://example.org/recipes#> SELECT (COUNT(?r) AS ?n) { ?r a rec:Recipe }',
                                graphs=DATA).n[0])

omelette = {'id': 'R15', 'name': 'Masala Omelette', 'name_fr': 'Omelette masala', 'cuisine': 'Indian',
            'prep_minutes': 15, 'servings': 1, 'created': '2025-12-01', 'description': 'Spiced omelette with onion.',
            'ingredients': [('egg', 3, 'piece'), ('onion', 1, 'piece')]}
salad = {'id': 'R16', 'name': 'Tomato Basil Salad', 'cuisine': 'Italian', 'prep_minutes': 10, 'servings': 2,
         'ingredients': [('tomato', 300, 'g'), ('basil', 10, 'leaf'), ('olive-oil', 20, 'ml')]}
stew = {'id': 'R17', 'name': 'Mystery Stew', 'cuisine': 'Martian', 'prep_minutes': 0, 'servings': 4,
        'ingredients': [('unicorn-meat', 1, 'kg'), ('onion', -2, 'piece')]}

print('recipes before:', recipe_count())
report = validation.load_validated(conn, batch(omelette, salad, stew), target=DATA, shapes=SHAPES)
print('batch accepted?', report.ok, '|', report.summary(), '| recipes after:', recipe_count())
report.results.assign(focus=report.results.focus.map(short))

recipes before: 14


batch accepted? False | 4 violation(s), 3 warning(s), 2 info(s) | recipes after: 14


,severity,focus,path,value,message,constraint
0,Violation,mystery-stew,hasIngredient,http://example.org/recipes/ingredient/unicorn-meat,"hasIngredient: at least one, and each must be a known rec:Ingredient",Class
1,Violation,mystery-stew,hasIngredientLine,(blank node),"hasIngredientLine: at least one, and each must be a valid ingredient line",Node
2,Violation,mystery-stew,hasIngredientLine,(blank node),"hasIngredientLine: at least one, and each must be a valid ingredient line",Node
3,Violation,mystery-stew,prepMinutes,0,"prepMinutes: exactly one whole number of minutes, 1-600",MinInclusive
4,Warning,mystery-stew,cuisine,Martian,"cuisine: not in the known list (a typo, or a new cuisine to add?)",In
5,Warning,mystery-stew,description,,description: missing,MinCount
6,Warning,tomato-basil-salad,description,,description: missing,MinCount
7,Info,mystery-stew,created,,created: no creation date,MinCount
8,Info,tomato-basil-salad,created,,created: no creation date,MinCount


**Rejected**: the stew breaks several rules, so the omelette and the salad (both fine) were **not** loaded either, and the recipe count is unchanged. The report tells the CMS team exactly what to fix. Two ingredient-line findings say only that a line is invalid (`Node` constraint); the details (`kg` isn't an allowed unit, `-2` isn't positive) come from validating a line on its own. Exercise 3 does that.

The CMS fixes the stew, and resends:

In [12]:
stew.update(id='R17', cuisine='French', prep_minutes=90, description='Slow-cooked root vegetable stew.',
            ingredients=[('carrot', 300, 'g'), ('potato', 400, 'g'), ('onion', 2, 'piece')])
report = validation.load_validated(conn, batch(omelette, salad, stew), target=DATA, shapes=SHAPES)
print('batch accepted?', report.ok, '|', report.summary(), '| recipes after:', recipe_count())
report.results.assign(focus=report.results.focus.map(short))

batch accepted? True | 0 violation(s), 1 warning(s), 2 info(s) | recipes after: 17


,severity,focus,path,value,message,constraint
0,Warning,tomato-basil-salad,description,,description: missing,MinCount
1,Info,mystery-stew,created,,created: no creation date,MinCount
2,Info,tomato-basil-salad,created,,created: no creation date,MinCount


**Accepted**: 3 recipes added. The remaining findings are the salad's missing description (Warning), and the missing creation dates of the salad and the stew (Info). They were reported, not blocking.

## Mini-project 2: an allergen-safe menu service

Level 2 ends by putting everything together. A restaurant needs to answer, for each guest:

> *"Which dishes can this guest eat, given their **allergies**, their **diet**, and how much **time** we have?"*

The pieces:

| Step | Chapter |
|---|---|
| New recipes enter only through the **validation gate** | 9 (done above) |
| **Allergens** per recipe come from the property chain | 7–8 |
| **Vegan** / **vegetarian** use the inferred category hierarchy | 8 |
| Everything runs as **`.rq` files** through `run_query` | 4 |

In [13]:
def reasoned(name):
    return sparql.run_query(conn, f'recipes/{name}', graphs=DATA, reasoning=True, schema=MODEL_NAME)

KNOWN_ALLERGENS = set(sparql.run_query(conn, 'PREFIX rec: <http://example.org/recipes#> SELECT ?n { ?a a rec:Allergen ; rec:name ?n }',
                                       graphs=DATA).n)
DIETS = {'any': None, 'vegetarian': 'vegetarian_recipes', 'vegan': 'vegan_recipes'}

def safe_menu(allergies=(), diet='any', max_minutes=None):
    '''Dishes a guest can eat. Allergies are names such as "Peanuts"; diet is any / vegetarian / vegan.'''
    unknown = set(allergies) - KNOWN_ALLERGENS
    if unknown or diet not in DIETS:
        raise ValueError(f'Unknown allergy {sorted(unknown)} or diet {diet!r}')   # never guess on allergies

    menu = sparql.run_query(conn, 'recipes/all_recipes', graphs=DATA)[['recipe', 'name', 'cuisine', 'minutes']]
    contains = reasoned('recipe_allergen_list')
    menu['allergens'] = menu.recipe.map(contains.groupby('recipe').allergen.apply(lambda a: ', '.join(sorted(a)))).fillna('')

    unsafe = set(contains[contains.allergen.isin(allergies)].recipe)
    menu = menu[~menu.recipe.isin(unsafe)]
    if DIETS[diet]:
        menu = menu[menu.recipe.isin(reasoned(DIETS[diet]).recipe)]
    if max_minutes is not None:
        menu = menu[menu.minutes <= max_minutes]
    return menu.drop(columns='recipe').reset_index(drop=True)

Design notes:

- Allergy names are checked against the **known list**. For allergies, a typo must be an **error**, never "no restriction".
- The query results are combined in pandas: three small, reusable `.rq` files beat one giant query.
- The recipes accepted by the gate (omelette, salad, stew) are included automatically.

Three guests:

In [14]:
from IPython.display import Markdown

guests = [
    ('Priya: allergic to peanuts and tree nuts, vegetarian, 30 minutes', dict(allergies=['Peanuts', 'Tree Nuts'], diet='vegetarian', max_minutes=30)),
    ('Tom: coeliac (no gluten)',                                         dict(allergies=['Gluten'])),
    ('Lea: vegan',                                                        dict(diet='vegan')),
]
for guest, wishes in guests:
    display(Markdown(f'**{guest}**'))
    display(safe_menu(**wishes))

**Priya: allergic to peanuts and tree nuts, vegetarian, 30 minutes**

,name,cuisine,minutes,allergens
0,Tomato Basil Salad,Italian,10,
1,Masala Omelette,Indian,15,Eggs
2,Lupin Flatbread,Mediterranean,20,Lupin
3,Pancakes,American,20,"Eggs, Gluten, Milk"
4,Margherita Pizza,Italian,30,"Gluten, Milk"
5,Potato Salad,German,30,"Celery, Eggs, Mustard"


**Tom: coeliac (no gluten)**

,name,cuisine,minutes,allergens
0,Tomato Basil Salad,Italian,10,
1,Masala Omelette,Indian,15,Eggs
2,Grilled Salmon,Mediterranean,20,Fish
3,Lupin Flatbread,Mediterranean,20,Lupin
4,Moules Marinieres,French,25,"Milk, Molluscs, Sulphites"
5,Pad Thai,Thai,25,"Crustaceans, Eggs, Fish, Peanuts, Soybeans"
6,Potato Salad,German,30,"Celery, Eggs, Mustard"
7,Thai Green Curry,Thai,35,"Crustaceans, Fish"
8,Vegetable Soup,French,40,Celery
9,Mystery Stew,French,90,


**Lea: vegan**

,name,cuisine,minutes,allergens
0,Tomato Basil Salad,Italian,10,
1,Lupin Flatbread,Mediterranean,20,Lupin
2,Vegetable Soup,French,40,Celery
3,Mystery Stew,French,90,


And a mistake the service must **not** tolerate:

In [15]:
try:
    safe_menu(allergies=['Peanut'])          # singular: not a known allergen name
except ValueError as e:
    print('Refused:', e)

Refused: Unknown allergy ['Peanut'] or diet 'any'


**"Why isn't Pad Thai on Priya's menu, and Carbonara on Tom's?"** Ask the reasoner for the proof:

In [16]:
print(reasoning.format_proof(reasoning.explain(conn, RECIPE['pad-thai'], REC.containsAllergen, ALLERGEN['peanuts'], MODEL_NAME)))
print()
print(reasoning.format_proof(reasoning.explain(conn, RECIPE['spaghetti-carbonara'], REC.containsAllergen, ALLERGEN['gluten'], MODEL_NAME)))

INFERRED: recipe:pad-thai rec:containsAllergen allergen:peanuts
    ASSERTED: rec:containsAllergen owl:propertyChainAxiom ( rec:hasIngredient rec:hasAllergen )
    ASSERTED: ingredient:peanuts rec:hasAllergen allergen:peanuts
    ASSERTED: recipe:pad-thai rec:hasIngredient ingredient:peanuts



INFERRED: recipe:spaghetti-carbonara rec:containsAllergen allergen:gluten
    ASSERTED: rec:containsAllergen owl:propertyChainAxiom ( rec:hasIngredient rec:hasAllergen )
    ASSERTED: ingredient:spaghetti rec:hasAllergen allergen:gluten
    ASSERTED: recipe:spaghetti-carbonara rec:hasIngredient ingredient:spaghetti


Pad Thai contains peanuts, and Carbonara's spaghetti carries gluten. The proofs say so in terms the kitchen can check, down to the individual ingredient. (Pad Thai has no gluten, so it **is** on Tom's menu.)

🎉 **Level 2 complete.** You can change data safely, write advanced queries, model a domain in OWL, reason over it, and guard it with SHACL.

## 9.8 Exercises

**Exercise 1.** Add a shape for **categories**: every `rec:Category` needs exactly one name and at most one `skos:broader`, which must be a `rec:Category`. Load it into `SHAPES` and validate.

<details><summary>Solution</summary>

```python
extra = b'''@prefix sh: <http://www.w3.org/ns/shacl#> . @prefix rec: <http://example.org/recipes#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
rec:CategoryShape a sh:NodeShape ; sh:targetClass rec:Category ;
  sh:property [ sh:path rec:name ; sh:minCount 1 ; sh:maxCount 1 ] ;
  sh:property [ sh:path skos:broader ; sh:maxCount 1 ; sh:class rec:Category ] .'''
with client.transaction(conn):
    conn.add(stardog.content.Raw(extra, 'text/turtle'), graph_uri=SHAPES)
print(validation.validate(conn, SHAPES, [DATA]).summary())
```
</details>

**Exercise 2.** Write a `sh:sparql` constraint: a recipe's **servings** must not exceed **4 × its number of ingredients**. Test it by setting Pesto Pasta's servings to 40.

<details><summary>Solution</summary>

```python
extra = b'''@prefix sh: <http://www.w3.org/ns/shacl#> . @prefix rec: <http://example.org/recipes#> .
rec:ServingsShape a sh:NodeShape ; sh:targetClass rec:Recipe ;
  sh:sparql [ a sh:SPARQLConstraint ; sh:message "too many servings for so few ingredients" ;
    sh:select \"\"\"PREFIX rec: <http://example.org/recipes#>
      SELECT $this ?value WHERE {
        { SELECT $this (COUNT(?i) AS ?n) WHERE { $this rec:hasIngredient ?i } GROUP BY $this }
        $this rec:servings ?value  FILTER(?value > 4 * ?n) }\"\"\" ] .'''
with client.transaction(conn):
    conn.add(stardog.content.Raw(extra, 'text/turtle'), graph_uri=SHAPES)
conn.update(PREFIXES + f'WITH <{DATA}> DELETE {{ recipe:pesto-pasta rec:servings ?s }} INSERT {{ recipe:pesto-pasta rec:servings 40 }} WHERE {{ recipe:pesto-pasta rec:servings ?s }}')
print(validation.validate(conn, SHAPES, [DATA]).results)
reload_data()
```
</details>

**Exercise 3.** The gate reported "each must be a valid ingredient line" for the stew. Find the exact reason by validating the lines **directly**: load the bad stew into a scratch graph, add a shape that targets `rec:IngredientLine` with `sh:node rec:IngredientLineShape`, and validate.

<details><summary>Solution</summary>

```python
SCRATCH = f'{PREFIX}:scratch'
bad = {'id': 'R17', 'name': 'Mystery Stew', 'cuisine': 'Martian', 'prep_minutes': 0, 'servings': 4,
       'ingredients': [('unicorn-meat', 1, 'kg'), ('onion', -2, 'piece')]}
extra = b'''@prefix sh: <http://www.w3.org/ns/shacl#> . @prefix rec: <http://example.org/recipes#> .
rec:LineTarget a sh:NodeShape ; sh:targetClass rec:IngredientLine ; sh:node rec:IngredientLineShape .'''
with client.transaction(conn):
    conn.clear(graph_uri=SCRATCH)
    conn.add(batch(bad), graph_uri=SCRATCH)
    conn.add(stardog.content.Raw(extra, 'text/turtle'), graph_uri=SHAPES)
print(validation.validate(conn, SHAPES, [SCRATCH, DATA]).results[['path', 'value', 'message']])
```
</details>

**Exercise 4.** Add a `diet='pescatarian'` option to `safe_menu` (fish and seafood allowed, no meat).

<details><summary>Solution</summary>

Write `recipes/pescatarian_recipes.rq` like `vegetarian_recipes.rq`, excluding only `category:meat`, then add it to `DIETS`:

```python
DIETS['pescatarian'] = 'pescatarian_recipes'
safe_menu(diet='pescatarian')
```
</details>

In [17]:
# your answers here

## 9.9 Clean up

In [18]:
CLEAN_UP = True

if CLEAN_UP:
    models.unregister_model(MODEL_NAME)
    graphs = sparql.run_query(conn, f'SELECT DISTINCT ?g {{ GRAPH ?g {{ ?s ?p ?o }} FILTER(STRSTARTS(STR(?g), "{PREFIX}")) }}').g.tolist()
    with client.transaction(conn):
        for g in graphs:
            conn.clear(graph_uri=g)
    print('Removed:', *graphs, sep='\n  ')
    print('Models now:', list(models.list_models()))
conn.close()

Removed:
  urn:tutorial:ch09:model
  urn:tutorial:ch09:shapes
  urn:tutorial:ch09:data
  urn:tutorial:ch09:demo:shapes
  urn:tutorial:ch09:demo:data


Models now: ['Recipe_Basic', 'Recipe_Python', 'SupplyChain']


## Summary

- **SHACL** validates data under the **closed world**: required values, cardinalities, datatypes, ranges, patterns, allowed lists, links to typed resources, and anything expressible in SPARQL.
- A **shape** targets focus nodes (`sh:targetClass`, …) and checks the values of **paths** with **constraints**. Give each one a `sh:message` and a **severity**.
- Stardog produces standard **validation reports**. `kg.validation.validate()` turns them into DataFrames and can validate **several graphs together**.
- `sh:conforms` fails on any finding. Pipelines usually block only on **Violations** (`Report.ok`).
- **The validation gate**: stage → validate with context → promote atomically or reject the whole batch.
- **Validate what comes in, reason over what's valid**: the allergen-safe menu combines the gate, inferred allergens and diets, and explanations.

**Next: Level 3, Chapter 10, Virtual Graphs.** Query data that stays in its source system (CSV, databases) through mappings, together with the graph.